<!--nav--> [🗺 Learning path](README.md) · **5/33** · ◀ [LoRA QLoRA FineTuning](./LoRA_QLoRA_FineTuning.ipynb) · [Distributed Training DeepSpeed](./Distributed_Training_DeepSpeed.ipynb) ▶

# Modern Full Fine-Tuning for Post-Training

## Why Full Fine-Tuning Still Matters

LoRA/QLoRA are memory-efficient, but full fine-tuning updates **every weight** in the model.
This means the model can learn **deeper patterns** across the entire sequence — not just low-rank approximations.

### When to use Full Fine-Tuning over LoRA

| Scenario | Best Method |
|----------|------------|
| Domain shift (medical, legal, code) | **Full FT** — needs deep adaptation |
| Limited data (<1K examples) | LoRA — less overfitting risk |
| Large model on small GPU | QLoRA — memory constrained |
| Maximum quality, no shortcuts | **Full FT** — best possible performance |
| Post-training alignment (SFT + DPO) | **Full FT** — learns full sequence patterns |

### What This Notebook Covers

```
Step 1: Setup & GPU detection
Step 2: Load & explore the dataset (UltraChat-200K)
Step 3: Data preprocessing — proper chat templates
Step 4: Full fine-tuning with modern techniques
         - Gradient checkpointing (save ~40% memory)
         - Cosine LR schedule with warmup
         - bf16 mixed precision
         - Proper weight decay
Step 5: Evaluate on TruthfulQA (Open LLM Leaderboard benchmark)
Step 6: Compare our model vs base model vs published scores
```

### The Post-Training Pipeline

```
Pre-trained model (knows language, facts, patterns)
        |
        v
Full Fine-Tuning on conversations (SFT)
        |
        | Updates ALL 1.1B weights
        | Model learns full sequence dependencies
        | Every attention head adapts to instruction-following
        v
Post-trained model (follows instructions, gives helpful answers)
        |
        v
Benchmark on TruthfulQA (measures truthfulness + helpfulness)
```

### Benchmark: TruthfulQA

TruthfulQA is one of the 4 benchmarks on the **Open LLM Leaderboard**.
It tests whether a model gives **truthful** answers vs repeating common misconceptions.

| Model | TruthfulQA (MC2) | Source |
|-------|------------------|--------|
| TinyLlama-1.1B (base) | ~37% | Open LLM Leaderboard |
| TinyLlama-1.1B-Chat | ~38% | Open LLM Leaderboard |
| **Our Full FT (this notebook)** | **target: 39-42%** | We measure it here |
| Llama-2-7B | ~39% | Open LLM Leaderboard |
| Mistral-7B | ~42% | Open LLM Leaderboard |

Even a small bump on a 1.1B model is meaningful — we're competing with models 6x larger.

---
**Runtime:** T4 GPU (Kaggle or Colab) — full FT of 1.1B fits in 15GB with gradient checkpointing

## Step 1: Install & Check GPU

In [ ]:
!pip install -q transformers trl datasets accelerate lm-eval

In [ ]:
import torch
import gc
import time
import json

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU Memory: %.1f GB" % total_memory)
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

def gpu_report(label):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print("%s -> Allocated: %.2f GB | Reserved: %.2f GB" % (label, used, reserved))

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Step 2: Load Dataset — UltraChat 200K

UltraChat is a high-quality multi-turn conversation dataset used to train models like Zephyr.
It's one of the best open datasets for post-training SFT.

**Why UltraChat over Alpaca?**
- Multi-turn conversations (not just single instruction-output)
- Higher quality responses (GPT-4 generated, human filtered)
- Diverse topics: writing, coding, reasoning, math, science
- The model sees the **full conversation flow**, not isolated Q&A

In [ ]:
from datasets import load_dataset

# UltraChat 200K — the SFT dataset behind Zephyr
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset = dataset.shuffle(seed=42).select(range(5000))  # 5K examples for demo

print("Dataset: %d conversations" % len(dataset))
print("\nColumns:", dataset.column_names)
print("\nSample conversation:")
for msg in dataset[0]["messages"][:3]:
    role = msg["role"]
    content = msg["content"][:150]
    print("  [%s]: %s..." % (role, content))

## Step 3: Data Preprocessing

Proper tokenization is critical for full fine-tuning. Unlike LoRA where we're just nudging the model,
full FT updates every weight — bad data formatting = bad model.

### Key decisions:
- **Chat template:** Use the model's native template (TinyLlama uses ChatML)
- **Sequence length:** 1024 tokens — long enough for multi-turn conversations
- **Label masking:** Only compute loss on assistant responses (not user prompts)
- **Packing:** Concatenate short conversations to fill sequences efficiently

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 1024  # longer sequences = model learns full conversation flow

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Format conversations using the model's chat template
def format_conversation(example):
    """Apply chat template to multi-turn conversations."""
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

formatted_dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)

# Analyze token lengths to understand our data
sample_lengths = []
for i in range(min(500, len(formatted_dataset))):
    tokens = tokenizer(formatted_dataset[i]["text"])["input_ids"]
    sample_lengths.append(len(tokens))

import numpy as np
print("Token length statistics (sample of 500):")
print("  Min: %d" % min(sample_lengths))
print("  Median: %d" % int(np.median(sample_lengths)))
print("  Mean: %d" % int(np.mean(sample_lengths)))
print("  Max: %d" % max(sample_lengths))
print("  Sequences > %d tokens: %d%%" % (MAX_LENGTH, 100 * sum(1 for l in sample_lengths if l > MAX_LENGTH) / len(sample_lengths)))
print("\nMax length set to %d — captures most conversations fully" % MAX_LENGTH)
print("\nSample formatted text (first 300 chars):")
print(formatted_dataset[0]["text"][:300])

## Step 4: Full Fine-Tuning

This is where full FT shines — every weight gets updated, so the model can learn
**deep patterns** across the full sequence:

### Memory Budget on T4 (15 GB)

```
Model weights (bf16):     ~2.2 GB
Gradients (bf16):         ~2.2 GB
Optimizer (AdamW, fp32):  ~8.8 GB  (2x fp32 copies: momentum + variance)
Activations:              ~1-3 GB  (with gradient checkpointing)
─────────────────────────────────
Total:                    ~14-16 GB  → tight fit on T4!
```

### Modern Techniques We Use

| Technique | What it does | Memory savings |
|-----------|-------------|----------------|
| **Gradient checkpointing** | Recomputes activations instead of storing them | ~40% less activation memory |
| **bf16 mixed precision** | Half-precision for forward/backward, fp32 for optimizer | ~50% less model memory |
| **Cosine LR schedule** | Smoothly decays learning rate → better convergence | N/A (quality) |
| **Weight decay** | L2 regularization → prevents overfitting | N/A (quality) |
| **Warmup steps** | Gradually increases LR → stable early training | N/A (quality) |

In [ ]:
from transformers import AutoModelForCausalLM, TrainerCallback
from trl import SFTConfig, SFTTrainer

clear_gpu()

# Load model — full precision weights, all trainable
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
gpu_report("After loading model")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total params: %dM" % (total_params / 1e6))
print("Trainable: %dM (%.1f%%) — ALL weights!" % (trainable_params / 1e6, 100.0 * trainable_params / total_params))

In [ ]:
class MemoryTracker(TrainerCallback):
    """Track peak GPU memory and loss during training."""
    def __init__(self):
        self.peak_mb = 0
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append((state.global_step, logs["loss"]))
        mem = torch.cuda.max_memory_allocated() / 1e6
        if mem > self.peak_mb:
            self.peak_mb = mem

tracker = MemoryTracker()

# Training config — modern best practices for full fine-tuning
training_args = SFTConfig(
    output_dir="./full_ft_output",
    
    # Training duration
    num_train_epochs=2,              # 2 epochs — enough for 5K examples
    
    # Batch size — small due to full FT memory requirements
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch size = 16
    
    # Learning rate — lower than LoRA since we update ALL weights
    learning_rate=2e-5,              # 10x lower than LoRA's typical 2e-4
    lr_scheduler_type="cosine",      # smooth decay → better final quality
    warmup_ratio=0.1,               # 10% warmup → stable start
    weight_decay=0.01,              # prevent overfitting (L2 regularization)
    
    # Memory optimization
    bf16=True,                       # bfloat16 mixed precision
    gradient_checkpointing=True,     # recompute activations to save memory
    
    # Sequence handling
    max_length=MAX_LENGTH,           # 1024 tokens for full conversations
    dataset_text_field="text",
    
    # Logging
    logging_steps=10,
    report_to="none",
    save_strategy="no",
    
    # Eval
    do_eval=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    callbacks=[tracker],
)

print("Training config:")
print("  Effective batch size: %d" % (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))
print("  Learning rate: %.1e (with cosine decay)" % training_args.learning_rate)
print("  Warmup: %.0f%% of steps" % (training_args.warmup_ratio * 100))
print("  Weight decay: %.2f" % training_args.weight_decay)
print("  Max sequence length: %d" % MAX_LENGTH)
print("  Gradient checkpointing: ON (saves ~40%% activation memory)")
print("  Precision: bf16")

In [ ]:
print("=" * 60)
print("  FULL FINE-TUNING — ALL %dM PARAMETERS" % (total_params / 1e6))
print("=" * 60)

start_time = time.time()
trainer.train()
train_time = time.time() - start_time

gpu_report("After training")
print("\nPeak GPU memory: %.2f GB" % (tracker.peak_mb / 1000))
print("Training time: %.1f seconds (%.1f minutes)" % (train_time, train_time / 60))

In [ ]:
# Save the fine-tuned model
trainer.save_model("./full_ft_output/final")
tokenizer.save_pretrained("./full_ft_output/final")
print("Model saved to ./full_ft_output/final")

## Step 5: Quick Generation Test

Before running the formal benchmark, let's see qualitative differences.

In [ ]:
# Test prompts — mix of factual, reasoning, and instruction-following
test_prompts = [
    "What causes lightning?",
    "Write a Python function that checks if a string is a palindrome.",
    "Explain the difference between TCP and UDP in simple terms.",
    "Is it true that we only use 10% of our brain?",  # TruthfulQA-style
    "What happens if you swallow gum?",                # TruthfulQA-style
]

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto",
)
base_model.eval()

# Our fine-tuned model
model.eval()

print("=" * 70)
print("  BASE MODEL vs FULL FINE-TUNED MODEL")
print("=" * 70)

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        # Base model response
        base_out = base_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        
        # Fine-tuned model response
        ft_out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
        ft_resp = tokenizer.decode(ft_out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    print("\nQ: %s" % prompt)
    print("  [BASE]:     %s" % base_resp[:250])
    print("  [FULL FT]:  %s" % ft_resp[:250])
    print("-" * 70)

# Clean up base model
del base_model
clear_gpu()

## Step 6: Benchmark — TruthfulQA (MC2)

TruthfulQA is part of the **Open LLM Leaderboard**. It measures whether models give
**truthful answers** instead of repeating popular misconceptions.

### How TruthfulQA MC2 Works

```
Question: "Can you teach old dogs new tricks?"

Correct answers (truthful):
  - "Yes, you can teach old dogs new tricks"          ✓
  - "Old dogs can learn new things"                   ✓

Incorrect answers (common misconception):
  - "No, you can't teach old dogs new tricks"         ✗
  - "Old dogs are unable to learn"                    ✗

Score: normalized probability of correct answers
MC2 = P(correct answers) / (P(correct) + P(incorrect))
```

We use the `lm-eval` harness — the same tool used by the Open LLM Leaderboard.

In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

print("=" * 60)
print("  BENCHMARK: TruthfulQA MC2")
print("  (Same benchmark used by Open LLM Leaderboard)")
print("=" * 60)

# Evaluate BASE model
print("\n--- Evaluating BASE model ---")
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto",
)
base_lm = HFLM(pretrained=base_model_eval, tokenizer=tokenizer)

base_results = lm_eval.simple_evaluate(
    model=base_lm,
    tasks=["truthfulqa_mc2"],
    batch_size=8,
)
base_score = base_results["results"]["truthfulqa_mc2"]["acc,none"]
print("Base model TruthfulQA MC2: %.4f (%.1f%%)" % (base_score, base_score * 100))

del base_model_eval, base_lm
clear_gpu()

In [ ]:
# Evaluate our FINE-TUNED model
print("\n--- Evaluating FULL FINE-TUNED model ---")
ft_lm = HFLM(pretrained=model, tokenizer=tokenizer)

ft_results = lm_eval.simple_evaluate(
    model=ft_lm,
    tasks=["truthfulqa_mc2"],
    batch_size=8,
)
ft_score = ft_results["results"]["truthfulqa_mc2"]["acc,none"]
print("Fine-tuned model TruthfulQA MC2: %.4f (%.1f%%)" % (ft_score, ft_score * 100))

del ft_lm

## Step 7: Results — Comparison with Open LLM Leaderboard

In [ ]:
from IPython.display import HTML, display

# Published benchmark scores from Open LLM Leaderboard
leaderboard = [
    ("GPT-J 6B", 0.380, "Open LLM Leaderboard"),
    ("TinyLlama-1.1B (base)", base_score, "Measured here"),
    ("TinyLlama-1.1B-Chat", 0.382, "Open LLM Leaderboard"),
    ("OUR Full FT (1.1B)", ft_score, "Measured here"),
    ("Llama-2-7B", 0.389, "Open LLM Leaderboard"),
    ("Phi-2 (2.7B)", 0.443, "Open LLM Leaderboard"),
    ("Mistral-7B", 0.425, "Open LLM Leaderboard"),
]

# Sort by score
leaderboard.sort(key=lambda x: x[1])

rows = ""
for name, score, source in leaderboard:
    is_ours = "OUR" in name
    is_measured = "Measured" in source
    color = "#3fb950" if is_ours else ("#58a6ff" if is_measured else "#8b949e")
    weight = "bold" if is_ours or is_measured else "normal"
    bg = "#1a2332" if is_ours else "transparent"
    
    bar_width = score * 100 * 2  # scale for visual
    bar_color = "#3fb950" if is_ours else "#30363d"
    
    rows += (
        '<tr style="background:%s;">'
        '<td style="padding:10px;color:%s;font-weight:%s;">%s</td>'
        '<td style="padding:10px;">'
        '<div style="background:#161b22;border-radius:4px;overflow:hidden;">'
        '<div style="width:%.0f%%;background:%s;padding:4px 8px;color:white;font-weight:600;font-size:13px;">%.1f%%</div>'
        '</div></td>'
        '<td style="padding:10px;color:#8b949e;font-size:12px;">%s</td>'
        '</tr>'
    ) % (bg, color, weight, name, min(score * 200, 100), bar_color, score * 100, source)

delta = ft_score - base_score
delta_str = "+%.1f%%" % (delta * 100) if delta > 0 else "%.1f%%" % (delta * 100)
delta_color = "#3fb950" if delta > 0 else "#f85149"

html = """
<div style="font-family: -apple-system, sans-serif; max-width: 800px; margin: 20px 0;">
  <h3 style="color: #c9d1d9;">TruthfulQA MC2 — Open LLM Leaderboard Comparison</h3>
  <table style="width:100%%; border-collapse:collapse; color:#c9d1d9;">
    <tr style="border-bottom:2px solid #30363d;">
      <th style="text-align:left; padding:10px; color:#a78bfa;">Model</th>
      <th style="text-align:left; padding:10px; color:#a78bfa;">TruthfulQA MC2</th>
      <th style="text-align:left; padding:10px; color:#a78bfa;">Source</th>
    </tr>
    %s
  </table>
  <div style="margin-top:16px; padding:16px; background:#161b22; border:1px solid #30363d; border-radius:8px;">
    <span style="color:#8b949e;">Improvement over base: </span>
    <span style="color:%s; font-weight:bold; font-size:18px;">%s</span>
    <span style="color:#8b949e;"> | Base: %.1f%% → Fine-tuned: %.1f%%</span>
  </div>
</div>
""" % (rows, delta_color, delta_str, base_score * 100, ft_score * 100)

display(HTML(html))

## Step 8: Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
if tracker.losses:
    steps = [x[0] for x in tracker.losses]
    losses = [x[1] for x in tracker.losses]
    axes[0].plot(steps, losses, color="#818cf8", linewidth=2, marker="o", markersize=3)
    axes[0].set_title("Training Loss (Full Fine-Tuning)", fontsize=14)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.2)

# Benchmark comparison bar chart
models_to_plot = [
    ("TinyLlama\n(base)", base_score, "#6e7681"),
    ("TinyLlama\n-Chat", 0.382, "#8b949e"),
    ("Our Full FT", ft_score, "#3fb950"),
    ("Llama-2\n-7B", 0.389, "#58a6ff"),
    ("Mistral\n-7B", 0.425, "#d2a8ff"),
]

names = [m[0] for m in models_to_plot]
scores = [m[1] for m in models_to_plot]
colors = [m[2] for m in models_to_plot]

bars = axes[1].bar(names, [s * 100 for s in scores], color=colors, edgecolor="white", linewidth=0.5)
axes[1].set_title("TruthfulQA MC2 (%%)", fontsize=14)
axes[1].set_ylabel("Accuracy (%%)")
axes[1].set_ylim(30, 50)
for bar, score in zip(bars, scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 "%.1f%%" % (score * 100), ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## Summary

In [ ]:
from IPython.display import HTML, display

html = """
<div style="font-family: -apple-system, sans-serif; max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #a78bfa; font-size: 13px; font-weight: 600;">Parameters Trained</div>
      <div style="color: #f0883e; font-size: 28px; font-weight: 700; margin: 8px 0;">%dM</div>
      <div style="color: #8b949e; font-size: 11px;">100%% of model (full FT)</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #a78bfa; font-size: 13px; font-weight: 600;">Peak GPU Memory</div>
      <div style="color: #58a6ff; font-size: 28px; font-weight: 700; margin: 8px 0;">%.1f GB</div>
      <div style="color: #8b949e; font-size: 11px;">T4 limit: 15 GB</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #a78bfa; font-size: 13px; font-weight: 600;">TruthfulQA MC2</div>
      <div style="color: #3fb950; font-size: 28px; font-weight: 700; margin: 8px 0;">%.1f%%%%</div>
      <div style="color: #8b949e; font-size: 11px;">vs base: %.1f%%%%</div>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
    <table style="width:100%%; color: #c9d1d9; font-size: 13px; border-spacing: 0 8px;">
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">TinyLlama 1.1B</td></tr>
      <tr><td style="color:#8b949e;">Method</td><td style="text-align:right; color:#f0883e;">Full Fine-Tuning (all weights)</td></tr>
      <tr><td style="color:#8b949e;">Dataset</td><td style="text-align:right;">UltraChat 200K (5K subset)</td></tr>
      <tr><td style="color:#8b949e;">Sequence Length</td><td style="text-align:right;">%d tokens</td></tr>
      <tr><td style="color:#8b949e;">Training Time</td><td style="text-align:right;">%.0f seconds</td></tr>
      <tr><td style="color:#8b949e;">Benchmark</td><td style="text-align:right; color:#3fb950;">TruthfulQA MC2 (Open LLM Leaderboard)</td></tr>
    </table>
  </div>
</div>
""" % (total_params / 1e6, tracker.peak_mb / 1000, ft_score * 100, base_score * 100, MAX_LENGTH, train_time)

display(HTML(html))

---

## How It All Works — Full Fine-Tuning Deep Dive

### Why Full FT Learns Better Sequence Patterns

```
LoRA:    W_new = W_frozen + A @ B    (rank-16 update, ~1% of capacity)
Full FT: W_new = W_old + gradient     (full-rank update, 100% of capacity)

For understanding full conversations:
  - LoRA can adjust attention patterns at the surface level
  - Full FT can rewire how the model represents multi-turn context
  - Full FT can change embeddings, attention, MLP, and layer norms together
  - This coordinated update captures deeper sequence dependencies
```

### Modern Training Techniques Explained

**1. Gradient Checkpointing**
```
Normal:       Store ALL activations during forward pass → backward uses them
              Memory: O(layers * batch * seq_len * hidden_dim)

Checkpointing: Store activations at "checkpoints" (every few layers)
               Recompute others during backward pass
               Memory: O(sqrt(layers) * batch * seq_len * hidden_dim)
               Cost: ~30% more compute, ~40% less memory
```

**2. Cosine Learning Rate Schedule**
```
Step LR:   lr = 2e-5 -----> lr = 2e-5 -----> lr = 2e-6 (sudden drop)
Cosine LR: lr = 2e-5 ~~~> lr = 1.5e-5 ~~~> lr = 5e-6 ~~~> lr = 0 (smooth)

Cosine is better because:
  - No sudden jumps that destabilize training
  - Naturally explores broadly early, refines late
  - Standard in all modern LLM training (GPT-4, Llama, Mistral)
```

**3. Weight Decay (L2 Regularization)**
```
Without decay: weights can grow large → overfitting to training data
With decay:    W_new = W_old - lr * gradient - lr * decay * W_old
               Gently pushes weights toward zero → generalizes better
```

### Full FT vs LoRA vs QLoRA — When to Use What

| Criteria | Full FT | LoRA | QLoRA |
|----------|---------|------|-------|
| Quality ceiling | Highest | High | Good |
| GPU memory (1.1B) | ~14 GB | ~4 GB | ~2 GB |
| GPU memory (7B) | ~84 GB | ~30 GB | ~8 GB |
| Training speed | Slowest | Fast | Fastest |
| Overfitting risk | Higher (more params) | Lower | Lowest |
| Best for | Domain adaptation, max quality | General fine-tuning | Large models on small GPUs |
| Data needed | More (>1K examples) | Less | Less |

### Platform Guide

| Platform | GPU | Full FT max model | Cost |
|----------|-----|-------------------|------|
| **Kaggle** | T4 (15 GB) | ~1.5B | Free (30h/week) |
| **Colab Free** | T4 (15 GB) | ~1.5B | Free |
| **Colab Pro** | A100 (40 GB) | ~3B | ~$10/mo |
| **Lambda/Paperspace** | A100 (80 GB) | ~7B | ~$1-2/hr |